# PD-L1: MAGE 20 + LoRA 20 + 양성 대조군 2쌍
런타임 → 런타임 유형 변경 → GPU를 선택합니다. 모든 서열·시드·코드는 이 노트북에 고정돼 있습니다.
Colab/공식 MMseqs2로 서열 전송은 사용자가 2026-11-14 승인했습니다. 유료 상품 가입·구매는 하지 않습니다.
출력은 결합 예측이 아닌 Fv/표적 구조이며, 이후 HADDOCK3 및 PRODIGY 단계가 필요합니다.


In [ ]:
import os, sys, subprocess, pathlib, json, time
from pathlib import Path
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout)
assert gpu.returncode == 0, '런타임 유형을 GPU로 변경한 후 실행하세요. CPU로 계산하지 않습니다.'
ENV = Path('/content/pdl1-env')
subprocess.run([sys.executable, '-m', 'venv', str(ENV/'colabfold')], check=True)
python = ENV/'colabfold/bin/python'
subprocess.run([str(python), '-m', 'pip', 'install', 'colabfold[alphafold,openmm]==1.6.3', 'jax[cuda12]==0.10.1'], check=True)
subprocess.run([str(python), '-c', 'import jax; print(jax.devices()); assert jax.default_backend()=="gpu"'], check=True)


In [ ]:
import base64, io, zipfile, hashlib
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'b717c8c3c6dcca19b3a2bafa1f51e001012eb7458a4324634470e05d6157001d'
ROOT = Path('/content/pdl1-project')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    archive.extractall(ROOT)
OUTPUT = ROOT/'results'
os.environ['PDL1_STRUCTURE_OUTPUT'] = str(OUTPUT)
os.environ['PDL1_STRUCTURE_ENV'] = str(ENV)
(OUTPUT/'colab_gpu.txt').write_text(gpu.stdout)
with (OUTPUT/'colab_requirements_lock.txt').open('w') as handle:
    subprocess.run([str(python), '-m', 'pip', 'freeze'], stdout=handle, check=True)
print('고정 입력: Fv 42쌍 + 공통 PD-L1, 시드 42; 정확한 코드 SHA256:', hashlib.sha256((ROOT/'구조평가.py').read_bytes()).hexdigest())


In [ ]:
# 같은 셀을 재실행해도 완료된 구조는 건너뜁니다. 먼저 1개씩 실제 속도를 확인합니다.
subprocess.run([str(python), str(ROOT/'구조평가.py'), 'fold', '--ids', 'PDL1_ECD'], check=True)
subprocess.run([str(python), str(ROOT/'구조평가.py'), 'fold-batch'], check=True)
subprocess.run([str(python), str(ROOT/'구조평가.py'), 'fold-summary'], check=True)


In [ ]:
# 모든 결과/로그를 로컬로 내려받아 HADDOCK3 단계에서 재사용합니다.
import shutil
from google.colab import files
archive = shutil.make_archive('/content/PDL1_ColabFold_results_seed42', 'zip', OUTPUT)
print('결과:', archive)
files.download(archive)
